# Train the Transformer

This notebook trains the same model as `train.py`. The selected TOML experiment uses byte-level BPE and is optimized for Apple Silicon with BF16 mixed precision, length-bucketed dynamic padding, pre-tokenization, native LayerNorm, and the fastest measured MPS attention path.

In [ ]:
from config import load_config
from train import train_model

config = load_config("configs/en_it_bpe.toml")
config

## Choose the run settings

`num_epochs` is the total target epoch, not the number of additional epochs. The config determines the dataset, tokenizer strategy, vocabulary size, and separate artifact paths. Set `preload` to `None` only when you intentionally want to start the configured experiment from scratch.

In [ ]:
config["num_epochs"] = 20
config["batch_size"] = 64
config["lr"] = 3e-4
config["precision"] = "auto"  # BF16 on Apple MPS
config["preload"] = "latest"
config

## Train

This is the long-running cell. `device="auto"` chooses CUDA first, then Apple MPS, then CPU. The progress bar reports loss, average loss, and the current dynamic source/target lengths. A checkpoint is saved before validation after every epoch. Validation then measures teacher-forced loss over the full split and beam-decodes fixed short, medium, and long examples with a KV cache.

In [ ]:
train_model(config, requested_device="auto")

## Inspect the learning curves

TensorBoard reads the training loss and validation metrics written by `train_model`. Stop the TensorBoard process from the notebook UI when you are finished.

In [ ]:
from config import get_experiment_path

log_dir = str(get_experiment_path(config))
%load_ext tensorboard
%tensorboard --logdir $log_dir